# Exact GPU BFS: LRX / Rapaport M2 pair sweep

Enable **GPU (2×T4)** and **Internet**, then Run All. One pip command installs a checksummed native wheel from GitHub Releases; no loader, compilation or Kaggle Inputs are needed. Change the configuration to choose the graph family and `(n,r)` range. `r` is the multiplicity of the repeated symbol: start `[0,...,n-r] + [n-r]*(r-1)`.

The graph definition is built on CPU; BFS runs on all visible GPUs. Workers persist across pairs, and eligible tuning profiles are reused with resource checks. Small orbits avoid expensive calibration. This is bounded tuning, not a guarantee of a globally optimal profile.

Output: every completed layer count/time, complete per-pair reports/profiles/runtime identities and compact terminal states. The last committed layer is sampled up to 1000 states; a previous layer is retained when strictly smaller than 1000. Full graph states/edges are not archived. `INCOMPLETE` is a partial traversal; its depth is not an exact eccentricity. The final depth of a complete traversal is the canonical start's eccentricity (a diameter only when that implication is justified).

Default grid is small and was selected for a quick reproducible test. Raising `N_MAX` can exceed VRAM or time limits. Pending pairs remain explicit. Rerun with the same output/configuration to reuse completed reports; this does not resume an unfinished frontier.

The package automatically chooses compatible precompiled SM75/80/86/89/90/100/103/120 code. This notebook delivery was executed on 2xT4; other SM components have compilation/inventory evidence, not execution acceptance here. Source and architecture: [GitHub](https://github.com/TryDotAtwo/MultiGPUBFS). Results remain under /kaggle/working/pair-sweep/FAMILY. Download the Kaggle output or use the same files on your own Linux GPU host.

For n=2, CayleyPy lrx() rejects the argument; this notebook explicitly extends the same L/R/X permutations, all three equal to [1,0]. M2 uses its original constructor. Results are stored under separate family directories.


For login-host preloading without GPU and offline shared-cache jobs, see [native prefetch](https://github.com/TryDotAtwo/MultiGPUBFS/blob/codex/integrated-bfs-best-practices-delivery/docs/native-prefetch.md). Ordinary online use remains automatic.

In [ ]:
%pip install "multigpubfs[cayleypy] @ https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261015-v20/multigpubfs-0.2.0.dev20-py3-none-any.whl#sha256=2a3213f46e02256dc3eeddf094a80488ceeddaba2f2bc70cebb85ee6c69ba7cb"


In [ ]:
print("IMPORT_START", flush=True)
import multigpubfs, cayleypy
multigpubfs.enable_native()
print("NATIVE_READY", flush=True)

# Change these settings, then Run All. Start with the small default grid.
FAMILY = "lrx"  # "lrx" or "rapaport_m2"
N_MIN, N_MAX = 2, 5
PAIR_SECONDS = 120
TOTAL_SECONDS = 600


In [ ]:
import csv, hashlib, json, time
from pathlib import Path
import torch
from multigpubfs.generic_session import persistent_graph_workers

root = Path("/kaggle/working/pair-sweep") / FAMILY
root.mkdir(parents=True, exist_ok=True)
configuration = dict(family=FAMILY, n_min=N_MIN, n_max=N_MAX,
                     pair_seconds=PAIR_SECONDS, total_seconds=TOTAL_SECONDS)
assert FAMILY in ("lrx", "rapaport_m2")
assert 2 <= N_MIN <= N_MAX and PAIR_SECONDS > 0 and TOTAL_SECONDS > 0
assert torch.cuda.device_count() > 0, "Enable GPU in Kaggle Settings"
constructor = getattr(cayleypy.PermutationGroups, FAMILY)
config_path = root / "configuration.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == configuration, "Use a new output directory for changed settings"
else:
    config_path.write_text(json.dumps(configuration, indent=2))
rows, layers = [], []
started = time.monotonic()

# One resident rank pool for the sweep; admission still checks actual resources.
with persistent_graph_workers():
    for n in range(N_MIN, N_MAX + 1):
        for r in range(1, n + 1):
            remaining = TOTAL_SECONDS - (time.monotonic() - started)
            case = root / f"n{n}-r{r}"
            report_path = case / "report.json"
            if report_path.exists():
                report = json.loads(report_path.read_text())
                assert report["status"] in ("COMPLETE", "INCOMPLETE")
                wall = None  # Reused receipt: do not invent elapsed time.
            elif remaining < 10:
                rows.append(dict(n=n, r=r, status="PENDING", eccentricity=None,
                                 completed_depth=None, states=None, api_seconds=None,
                                 bfs_seconds=None, tuning_seconds=None))
                continue
            else:
                # CayleyPy lrx() asserts n>=3; extend the same L/R/X definition at n=2.
                definition = (cayleypy.CayleyGraphDef.create(
                    [[1, 0], [1, 0], [1, 0]], central_state=[0, 1],
                    generator_names=["L", "R", "X"], name="lrx-2")
                    if FAMILY == "lrx" and n == 2 else constructor(n))
                graph = cayleypy.CayleyGraph(definition, device="cpu")
                start = torch.tensor(list(range(n-r+1)) + [n-r]*(r-1), dtype=torch.int64)
                print("PAIR_START", n, r, flush=True)
                at = time.monotonic()
                result = graph.bfs(start_states=start, output=case,
                    native_options=multigpubfs.NativeOptions(
                        max_seconds=max(1, int(min(PAIR_SECONDS, remaining))), autotune=True))
                wall = time.monotonic() - at
                report = result.report
            sizes = report["layer_sizes"]
            depth = len(sizes) - 1
            rows.append(dict(n=n, r=r, status=report["status"],
                eccentricity=depth if report["status"] == "COMPLETE" else None,
                completed_depth=depth, states=sum(sizes), api_seconds=wall,
                bfs_seconds=report.get("bfs_seconds"),
                tuning_seconds=report.get("autotune", {}).get("seconds")))
            times = report.get("layer_seconds", [])
            cumulative = 0
            for d, size in enumerate(sizes):
                cumulative += size
                layers.append(dict(n=n, r=r, depth=d, states=size,
                    cumulative_states=cumulative,
                    expansion_seconds=times[d-1] if d > 0 and d-1 < len(times) else None))
            print(n, r, report["status"], depth, sum(sizes), flush=True)

for name, records in (("results.csv", rows), ("layers.csv", layers)):
    with (root/name).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(records[0]) if records else [])
        writer.writeheader(); writer.writerows(records)
manifest = {str(f.relative_to(root)): hashlib.sha256(f.read_bytes()).hexdigest()
            for f in sorted(root.rglob("*")) if f.is_file() and f.name != "checksums.json"}
(root/"checksums.json").write_text(json.dumps(manifest, indent=2))
import pandas as pd
display(pd.DataFrame(rows))
print("Saved:", root, "GPU count:", torch.cuda.device_count())
